# CaptionBERT four-million-row follow-up — seed 1

Use **G4 / RTX PRO6000** and enable notebook access to **HF_TOKEN**. Run setup,
the full-batch gate and TensorBoard, then the larger experiments and frozen
follow-up. The two training cells run sequentially; queue the frozen cell
behind the optimizer cell. Keep the v0.3.0 training environment.

This notebook restores the **locked, existing campaign**, including compatible
partial checkpoints. It verifies completed runs before skipping them. It runs
full4M hierarchical, full4M relative, then the hierarchical 1M matched-exposure
control, all under seed 1. Do not run another notebook with this seed
concurrently. The other notebook owns seed 2.

The final frozen cell restores the earlier completed binding checkpoint even
after a fresh runtime. It preserves the original intact reports, then measures
frozen interventions, reconstructed initialization, safe AutoModel parity and
synthetic shapes through 8192. Shape checks do not establish 8K semantic quality.

Training stays pinned to **v0.3.0**. The helper and probe source revisions are
explicit inside their cells. Interrupted sessions repeat setup/gate/TensorBoard
and these cells; they do not rerun the original screen queue.

[Protocol](https://github.com/AbstractEyes/captionbert-trainer/blob/codex/structured-rotary-overnight/docs/structured_campaign_protocol.md)
· [Results and artifacts](https://huggingface.co/AbstractPhil/captionbert-8192-rotary)


In [ ]:
import os
import pathlib
import subprocess
import sys
import urllib.request

CONFIG = {"release": "v0.3.0", "lane": 0, "env": "/content/captionbert-structured-env"}
required = {"release", "lane", "env"}
if required - CONFIG.keys():
    raise ValueError("Incomplete setup configuration; restore this tagged notebook and rerun setup")
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Choose Colab Python3.11-3.13, then rerun this setup cell")
os.environ["HF_HOME"] = "/content/captionbert-hf"
os.environ["HF_HUB_CACHE"] = "/content/captionbert-hf/hub"
env_dir = pathlib.Path(CONFIG["env"])
python = str(env_dir / "bin/python")
if not pathlib.Path(python).exists():
    bootstrap = pathlib.Path("/content/captionbert-bootstrap")
    subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(bootstrap), "virtualenv>=20.38.0"], check=True)
    bootstrap_env = os.environ.copy()
    bootstrap_env["PYTHONPATH"] = str(bootstrap)
    subprocess.run([sys.executable, "-m", "virtualenv", "--no-download", str(env_dir)], env=bootstrap_env, check=True)
req_dir = pathlib.Path("/content/captionbert-structured-install")
req_dir.mkdir(exist_ok=True)
for name in ("requirements.txt", "requirements-cuda.txt", "requirements-colab.txt"):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/AbstractEyes/captionbert-trainer/{CONFIG['release']}/{name}", req_dir / name)
subprocess.run([python, "-m", "pip", "install", "-r", str(req_dir / "requirements-colab.txt")], check=True)
subprocess.run([python, "-m", "pip", "install", "--no-deps", f"git+https://github.com/AbstractEyes/captionbert-trainer.git@{CONFIG['release']}"], check=True)
subprocess.run([python, "-m", "pip", "check"], check=True)
subprocess.run([python, "-c", "import torch,numpy,scipy,transformers,datasets,safetensors,captionbert_rotary; from torch.utils.tensorboard import SummaryWriter; from captionbert_rotary.modeling_captionbert import CaptionBertModel; print('READY',captionbert_rotary.__version__,torch.__version__,torch.version.cuda); assert torch.cuda.is_available(), 'Select G4 via the arrow beside Connect -> Change runtime type'; print(torch.cuda.get_device_name(0)); assert '6000' in torch.cuda.get_device_name(0), 'This campaign requires the selected G4 / RTX PRO6000'; print('GPU GB',torch.cuda.get_device_properties(0).total_memory/1e9)"], check=True)
from google.colab import userdata
try:
    token = userdata.get("HF_TOKEN")
except Exception as error:
    raise RuntimeError("Enable HF_TOKEN notebook access in the Secrets sidebar, then rerun setup") from error
os.environ["HF_TOKEN"] = token
subprocess.run([python, "-c", "from huggingface_hub import HfApi; print('Hub access:',HfApi().whoami()['name'])"], check=True)
print("Setup ready; next run the full-batch engineering gate, TensorBoard, then this lane's training cell.", flush=True)


In [ ]:
import os
import pathlib
python = "/content/captionbert-structured-env/bin/python"
root = pathlib.Path("/content/captionbert-structured/screen")
root.mkdir(parents=True, exist_ok=True)
if not pathlib.Path(python).exists():
    raise RuntimeError("Run setup cell first; the isolated Python environment is missing")
"""Parent-process output relay for notebook cells and an on-disk console log."""


def run_logged(command, log_path, env=None):
    import os
    from pathlib import Path
    import subprocess
    import time

    child_env = os.environ.copy() if env is None else dict(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Streaming child output; console log: {path}", flush=True)
    started = time.monotonic()
    with path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   env=child_env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        print(f"Child exited with code {returncode}; elapsed {time.monotonic() - started:.1f}s; "
              f"console log: {path}", flush=True)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    return returncode

run_logged([python, "-m", "captionbert_rotary.structured_preflight", "--full", "--output", str(root / "preflight.json")], root / "preflight-console.log")
print("Full-width batch2048 x256 gate passed. Next start TensorBoard, then training.", flush=True)


In [ ]:
import os
import pathlib
import subprocess
from google.colab import output

root = pathlib.Path("/content/captionbert-structured")
root.mkdir(parents=True, exist_ok=True)
pid_path = root / "tensorboard.pid"
alive = False
if pid_path.exists():
    try:
        os.kill(int(pid_path.read_text()), 0)
        alive = True
    except (OSError, ValueError):
        pass
if not alive:
    log = (root / "tensorboard-console.log").open("a")
    process = subprocess.Popen(["/content/captionbert-structured-env/bin/python", "-m", "tensorboard.main", "--logdir", str(root), "--host", "127.0.0.1", "--port", "6006"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    pid_path.write_text(str(process.pid))
print("TensorBoard: each arm/seed has its own run. Scalar data appear as training starts.")
output.serve_kernel_port_as_iframe(6006, height=420)


In [ ]:
import os
import hashlib
import pathlib
import subprocess
import urllib.request
from datetime import datetime, timezone
from google.colab import userdata

CONFIG = {"lane": "A", "root": "/content/captionbert-structured", "data": "/content/captionbert-data", "selection_url": "https://huggingface.co/AbstractPhil/captionbert-8192-rotary/resolve/becdda8590d138a92e947976005357fd6ccb9de2/campaigns/structured-20261106/locked-screen-selection.json"}
root = pathlib.Path(CONFIG["root"])
python = "/content/captionbert-structured-env/bin/python"
selection = pathlib.Path("/content/captionbert-locked-selection.json")
source = pathlib.Path("/content/captionbert-final-stage.py")
urllib.request.urlretrieve(CONFIG["selection_url"], selection)
if hashlib.sha256(selection.read_bytes()).hexdigest() != "bdbf782e86b29545ec4a0731a8669b69ed5794f82914bebd1236fab007820338":
    raise RuntimeError("Locked selection hash differs; restore the complete published cell")
urllib.request.urlretrieve("https://raw.githubusercontent.com/AbstractEyes/captionbert-trainer/768bce0a1af71f785ddbcd4bb5a7ab36da39046b/tools/run_structured_final_stage.py", source)
env = os.environ.copy()
env.update(HF_TOKEN=userdata.get("HF_TOKEN"), HF_HOME="/content/captionbert-hf", HF_HUB_CACHE="/content/captionbert-hf/hub", OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
console = root / ("larger-" + CONFIG["lane"] + "-" + stamp + ".log")
command = [python, str(source), "--lane", CONFIG["lane"], "--selection", str(selection), "--root", str(root), "--data", CONFIG["data"]]
print("Starting the locked larger experiment on lane", CONFIG["lane"], "; original training release preserved.", flush=True)
print("Console:", console, flush=True)
with console.open("a", encoding="utf-8", buffering=1) as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1, env=env)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        returncode = process.wait()
    except KeyboardInterrupt:
        import signal
        process.send_signal(signal.SIGINT)
        raise
    finally:
        process.stdout.close()
print("Larger lane exit:", returncode, "; preserving its complete console.", flush=True)
subprocess.run(command + ["--ship-only", "--console", str(console)], env=env, check=True)
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print("Larger lane complete: all three seed-owned models and logs uploaded.", flush=True)


In [ ]:
import os
import json
import pathlib
import subprocess
import urllib.request
from datetime import datetime, timezone
from google.colab import userdata

CONFIG = {"lane": "A", "root": "/content/captionbert-structured", "data": "/content/captionbert-data", "source": "59ec95344958120a328f65a0b056632db66ee088"}
root = pathlib.Path(CONFIG["root"])
seed = {"A": 1, "B": 2}[CONFIG["lane"]]
expected = [("full4m", "H16", 4000000), ("full4m", "N16", 4000000), ("exposure1m", "H16", 1000000)]
for stage, arm, rows in expected:
    path = root / stage / f"{arm}-seed{seed}" / "result.json"
    if not path.is_file():
        raise RuntimeError("Finish or resume this lane's three larger models before running the frozen follow-up")
    result = json.loads(path.read_text())
    if result.get("status") != "complete" or result.get("steps") != 7808 or result.get("training_rows") != rows:
        raise RuntimeError("A larger model is incomplete or has a different exposure; inspect and resume its registered training cell")
python = "/content/captionbert-structured-env/bin/python"
source = pathlib.Path("/content/captionbert-final-frozen-probe.py")
urllib.request.urlretrieve("https://raw.githubusercontent.com/AbstractEyes/captionbert-trainer/" + CONFIG["source"] + "/tools/probe_structured_campaign.py", source)
env = os.environ.copy()
env.update(HF_TOKEN=userdata.get("HF_TOKEN"), HF_HOME="/content/captionbert-hf", HF_HUB_CACHE="/content/captionbert-hf/hub", OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
base = [python, str(source), "--data", CONFIG["data"], "--rows", "4096", "--batch", "8", "--push", "--output-name", "frozen-followup-" + CONFIG["source"][:12]]
jobs = [
    ["--root", str(root / "full4m"), "--arms", "H16", "N16", "--long-shapes", "--structural-ablations", "--sentence-ablations"],
    ["--root", str(root / "exposure1m"), "--arms", "H16", "--long-shapes", "--structural-ablations", "--sentence-ablations"],
    ["--root", str(root / "screen"), "--arms", "KALLP", "--ablations", "--sentence-ablations", "--initialization-control"],
]
restore = "import sys\nfrom pathlib import Path\nfrom captionbert_rotary.structured_campaign import CampaignConfig\nfrom captionbert_rotary.hub import pull_completed\ncfg = CampaignConfig(stage=\"screen\", arms=(\"KALLP\",), seeds=(int(sys.argv[2]),), output_dir=str(Path(sys.argv[1]) / \"screen\"), data_dir=sys.argv[3])\nif cfg.fingerprint() != \"dc37215529c188be7b93804c4a21f125afa353e037b1a2a8fc11e5b2fc817d3a\":\n    raise RuntimeError(\"Restore the original v0.3.0 training environment before recovering the screen checkpoint\")\npull_completed(cfg, \"659b21c1c6eb1074eeec7acdbfa9bd010965595a4669338690cf3ae38a4448d5\")\nif not (Path(cfg.output_dir) / f\"KALLP-seed{sys.argv[2]}\" / \"result.json\").is_file():\n    raise RuntimeError(\"The completed KALLP screen checkpoint is missing; restore it before running the frozen follow-up\")\nprint(\"Earlier completed binding checkpoint restored or already present; no training\", flush=True)\n"
commands = [[python, "-c", restore, str(root), str(seed), CONFIG["data"]]] + [base + options for options in jobs]
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
console = root / ("frozen-final-" + CONFIG["lane"] + "-" + stamp + ".log")
print("Frozen follow-up after all optimizer work; preserving original intact reports.", flush=True)
print("Console:", console, flush=True)
returncode = 0
with console.open("a", encoding="utf-8", buffering=1) as log:
    for command in commands:
        print("Frozen command:", command, flush=True)
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1, env=env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        if returncode:
            break
ship = "import sys; from huggingface_hub import HfApi; HfApi().upload_file(repo_id='AbstractPhil/captionbert-8192-rotary', path_or_fileobj=sys.argv[1], path_in_repo=sys.argv[2], commit_message='Preserve complete final frozen follow-up console')"
remote = "campaigns/structured-20261106/operations/frozen-final-" + CONFIG["lane"] + "/" + stamp + "/console.log"
subprocess.run([python, "-c", ship, str(console), remote], env=env, check=True)
if returncode:
    raise subprocess.CalledProcessError(returncode, command)
print("Final frozen reports, query arrays, TensorBoard scalars, long shapes, and console uploaded.", flush=True)
